# MCP End-to-End Workshop: HTTP → user_controller → MCP

This notebook walks through the complete request path using the **actual project source files**:

| File | Role |
|------|------|
| `src/db_server.py` | MCP server — resources (read) + tools (write) |
| `src/services/mcp_client.py` | MCP stdio client wrapper (`MCPDBClient`) |
| `src/api_server.py` | FastAPI application entry point |
| `src/controllers/user_controller.py` | HTTP router → `MCPDBClient` |
| `src/db_client.py` | AI agent CLI client (alternative consumer) |

## Session Plan

1. **Server Design** — explore `db_server.py`: resources vs tools
2. **MCP Client Layer** — explore `mcp_client.py`: stdio wrapper
3. **API & Controller Layer** — `api_server.py` + `user_controller.py`
4. **AI Agent Client** — `db_client.py`: LLM-powered alternative consumer
5. **Client Discovery** — list live MCP capabilities
6. **Infrastructure Validation** — ensure Docker + API are running
7. **Create User** — POST end-to-end through the full stack
8. **Fetch Created User** — GET via HTTP and directly via MCP resource
9. **Flow Diagram** — Mermaid sequence diagram of the entire path

## Part 1) Server Design: `db_server.py`

`src/db_server.py` is the MCP server built with `FastMCP`. It exposes:

- **Resources** (read-only): `db://users`, `db://users/{user_id}` — decorated with `@mcp.resource`, map to `SELECT` queries
- **Tools** (mutations): `create_user`, `update_user`, `delete_user` — decorated with `@mcp.tool`, map to `INSERT`/`UPDATE`/`DELETE`

The server runs via `mcp.run(transport="stdio")`, which means MCP clients spawn it as a subprocess and communicate over stdin/stdout.

In [6]:
from pathlib import Path

print(Path("src/db_server.py").read_text())

from typing import Any
import logging
import os

import asyncpg
from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

load_dotenv()

mcp = FastMCP("db_server")


def _db_config() -> dict[str, Any]:
    return {
        "host": os.getenv("POSTGRES_HOST", "127.0.0.1"),
        "port": int(os.getenv("POSTGRES_PORT", "5432")),
        "database": os.getenv("POSTGRES_DB", "mcp_lab"),
        "user": os.getenv("POSTGRES_USER", "mcp_user"),
        "password": os.getenv("POSTGRES_PASSWORD", "mcp_password"),
    }


async def _connect() -> asyncpg.Connection:
    return await asyncpg.connect(**_db_config())


async def ensure_schema() -> None:
    conn = await _connect()
    try:
        await conn.execute(
            """
            CREATE TABLE IF NOT EXISTS users (
                id SERIAL PRIMARY KEY,
                name TEXT NOT NULL,
                email TEXT NOT NULL UNIQUE,
                created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
            );
            """

## Part 2) MCP Client Layer: `mcp_client.py`

`src/services/mcp_client.py` contains `MCPDBClient`, which wraps the MCP stdio protocol.

Key design decisions:

- Each request **spawns a short-lived subprocess** (`db_server.py`) via `stdio_client`
- **`_read_resource(uri)`** calls MCP resources for read operations
- **`_call_tool(name, args)`** calls MCP tools for write operations
- Returns uniform `{success, message, data}` dicts so the controller layer never touches raw MCP types

In [7]:
from pathlib import Path

print(Path("src/services/mcp_client.py").read_text())

from __future__ import annotations

import json
import sys
from typing import Any

from mcp import ClientSession
from mcp.client.stdio import StdioServerParameters, stdio_client
from pydantic import AnyUrl, TypeAdapter


class MCPDBClient:
    def __init__(self) -> None:
        # Use the same Python runtime as the API process so MCP server dependencies match.
        self._server_params = StdioServerParameters(command=sys.executable, args=["src/db_server.py"])

    async def _with_session(self) -> ClientSession:
        raise RuntimeError("Use within async context manager")

    @staticmethod
    def _to_any_url(uri: str) -> AnyUrl:
        return TypeAdapter(AnyUrl).validate_python(uri)

    @staticmethod
    def _parse_json_text(text: str) -> dict[str, Any]:
        try:
            parsed = json.loads(text)
            if isinstance(parsed, dict):
                return parsed
            return {"success": False, "message": "Unexpected non-object response", "data": parsed}
       

## Part 3) API & Controller Layer: `api_server.py` + `user_controller.py`

`src/api_server.py` creates the FastAPI app and mounts the `/users` router.

`src/controllers/user_controller.py` bridges **HTTP ↔ MCP**:

- Receives HTTP requests on routes like `POST /users/create_user`
- Delegates directly to `mcp_db_client.*` (a module-level `MCPDBClient` instance)
- Maps MCP `{success: false}` payloads → HTTP `400`/`404`/`422` errors via `_raise_http_if_failed()`

In [8]:
from pathlib import Path

print("=== src/api_server.py ===\n")
print(Path("src/api_server.py").read_text())

print("\n" + "=" * 60)
print("=== src/controllers/user_controller.py ===\n")
print(Path("src/controllers/user_controller.py").read_text())

=== src/api_server.py ===

from fastapi import FastAPI

from src.controllers.user_controller import router as users_router

app = FastAPI(
    title="MCP Users API",
    description="Swagger API that routes user read/write operations through an MCP controller-client chain.",
    version="1.0.0",
)


@app.get("/health", tags=["Health"]) # type: ignore
async def health() -> dict[str, str]:
    return {"status": "ok"}


app.include_router(users_router)

=== src/controllers/user_controller.py ===

from typing import Any

from fastapi import APIRouter, HTTPException
from pydantic import BaseModel

from src.services.mcp_client import mcp_db_client

router = APIRouter(prefix="/users", tags=["Users"])


class CreateUserRequest(BaseModel):
    name: str
    email: str


class UpdateUserRequest(BaseModel):
    name: str
    email: str


def _raise_http_if_failed(payload: dict[str, Any], not_found_message: str | None = None) -> None:
    if payload.get("success") is True:
        return

    # Ma

## Part 4) Alternative Client: `db_client.py`

`src/db_client.py` is an **alternative MCP consumer** built with the **OpenAI Agents SDK** (`openai-agents`).

Instead of calling MCP directly, it wraps an LLM agent that interprets CLI commands and automatically selects the correct MCP tool or resource:

```
python src/db_client.py create --name Alice --email alice@example.com
python src/db_client.py list
python src/db_client.py get --user-id 1
```

> **Note**: Executing this client requires `OPENAI_API_KEY` set in `.env` and `pip install openai-agents`.  
> The cell below shows the source for reference — it is not executed.

In [9]:
from pathlib import Path

# Reference only — requires OPENAI_API_KEY + openai-agents to execute
print(Path("src/db_client.py").read_text())

import argparse
import asyncio

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, MCPServerStdioParams

load_dotenv()


def build_prompt(args: argparse.Namespace) -> str:
    if args.command == "create":
        return (
            "Use the create_user tool with the provided values and return the tool output only. "
            f"name={args.name}, email={args.email}."
        )

    if args.command == "update":
        return (
            "Use the update_user tool with the provided values and return the tool output only. "
            f"user_id={args.user_id}, name={args.name}, email={args.email}."
        )

    if args.command == "delete":
        return (
            "Use the delete_user tool and return the tool output only. "
            f"user_id={args.user_id}."
        )

    if args.command == "get":
        return (
            "Read the db://users/{user_id} resource and return the exact resource output only. "
    

## Part 5) Client Discovery: List MCP Capabilities

Connect directly to `src/db_server.py` as an MCP client and enumerate what the server exposes — resources, resource templates, and tools.

In [14]:
import json
import sys
from mcp import ClientSession
from mcp.client.stdio import StdioServerParameters, stdio_client


async def list_db_server_capabilities() -> None:
    params = StdioServerParameters(command=sys.executable, args=["src/db_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            resources = await session.list_resources()
            templates = await session.list_resource_templates()
            tools = await session.list_tools()

    # ── Resources ─────────────────────────────────────────────────────────────
    print("=" * 60)
    print("RESOURCES  (read-only, @mcp.resource)")
    print("=" * 60)
    for r in resources.resources:
        print(f"\n  URI:         {r.uri}")
        print(f"  Name:        {r.name}")
        print(f"  Description: {r.description or '—'}")
        print(f"  MIME type:   {r.mimeType or 'application/json'}")

    # ── Resource Templates ────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("RESOURCE TEMPLATES  (parametric resources)")
    print("=" * 60)
    for t in templates.resourceTemplates:
        print(f"\n  URI template: {t.uriTemplate}")
        print(f"  Name:         {t.name}")
        print(f"  Description:  {t.description or '—'}")
        print(f"  MIME type:    {t.mimeType or 'application/json'}")

    # ── Tools ─────────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("TOOLS  (mutations, @mcp.tool)")
    print("=" * 60)
    for t in tools.tools:
        schema = t.inputSchema or {}
        props  = schema.get("properties", {})
        req    = schema.get("required", [])

        print(f"\n  Tool:         {t.name}")
        print(f"  Description:  {t.description or '—'}")
        print(f"  Parameters:")
        if props:
            for param_name, param_schema in props.items():
                ptype    = param_schema.get("type", "any")
                ptitle   = param_schema.get("title", param_name)
                required = "required" if param_name in req else "optional"
                print(f"    - {param_name:<12} type={ptype:<8}  {required}  (title: {ptitle})")
        else:
            print("    (no parameters)")
        print(f"  Full JSON schema:")
        print("    " + json.dumps(schema, indent=2).replace("\n", "\n    "))


await list_db_server_capabilities()


RESOURCES  (read-only, @mcp.resource)

  URI:         db://users
  Name:        users_resource
  Description: —
  MIME type:   text/plain

RESOURCE TEMPLATES  (parametric resources)

  URI template: db://users/{user_id}
  Name:         user_resource
  Description:  —
  MIME type:    text/plain

TOOLS  (mutations, @mcp.tool)

  Tool:         create_user
  Description:  Create a user record in PostgreSQL.
  Parameters:
    - name         type=string    required  (title: Name)
    - email        type=string    required  (title: Email)
  Full JSON schema:
    {
      "properties": {
        "name": {
          "title": "Name",
          "type": "string"
        },
        "email": {
          "title": "Email",
          "type": "string"
        }
      },
      "required": [
        "name",
        "email"
      ],
      "title": "create_userArguments",
      "type": "object"
    }

  Tool:         update_user
  Description:  Update an existing user record.
  Parameters:
    - user_id     

## Part 6) Infrastructure Validation

Ensure the PostgreSQL Docker container is running and the FastAPI server (`src/api_server.py`) is reachable on port `8000`.

> If the API is not running, start it in a terminal: `uvicorn src.api_server:app --reload`

In [11]:
import socket
import subprocess
from pathlib import Path

import requests

repo_root = Path.cwd()
subprocess.run(["docker", "compose", "up", "-d"], cwd=repo_root, check=False)

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    sock.settimeout(1)
    api_up = sock.connect_ex(("127.0.0.1", 8000)) == 0

print("API reachable on 8000:", api_up)
if not api_up:
    print("Start the API server: uvicorn src.api_server:app --reload")
else:
    for path in ["/health", "/users/get_users"]:
        r = requests.get(f"http://127.0.0.1:8000{path}", timeout=10)
        print(f"{path}  →  {r.status_code}")
        print(r.text[:250])

 Container gen_ai_mcp_lab_postgres  Running


API reachable on 8000: True
/health  →  200
{"status":"ok"}
/users/get_users  →  200
{"success":true,"message":"Fetched users","data":{"users":[{"id":1,"name":"Notebook Demo User","email":"notebook.demo.20260314224138@example.com","created_at":"2026-03-14T22:41:39.304186Z"},{"id":2,"name":"Notebook Demo User","email":"notebook.demo.2


## Part 7) Create User

Send a `POST /users/create_user` request through the **full stack**:

```
HTTP Client
  → api_server.py  (FastAPI router)
  → user_controller.py  (create_user route)
  → mcp_client.py  (MCPDBClient.create_user)
  → db_server.py   (MCP tool: create_user)
  → PostgreSQL     (INSERT INTO users)
```

The created user's `id` is stored in `created_user_id` for the next cell.

In [12]:
from datetime import datetime, timezone

import requests

base = "http://127.0.0.1:8000"
unique = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

payload = {
    "name": "Workshop Demo User",
    "email": f"workshop.{unique}@example.com",
}

resp = requests.post(f"{base}/users/create_user", json=payload, timeout=10)
print("HTTP status:", resp.status_code)
print("Response:   ", resp.json())

create_data = resp.json()
created_user = create_data.get("data", {}).get("user", {})
created_user_id = created_user.get("id")

print(f"\nCreated user ID:    {created_user_id}")
print(f"Created user name:  {created_user.get('name')}")
print(f"Created user email: {created_user.get('email')}")

HTTP status: 200
Response:    {'success': True, 'message': 'User created', 'data': {'user': {'id': 7, 'name': 'Workshop Demo User', 'email': 'workshop.20260315180353@example.com', 'created_at': '2026-03-15T18:03:54.049474Z'}}}

Created user ID:    7
Created user name:  Workshop Demo User
Created user email: workshop.20260315180353@example.com


## Part 8) Fetch Created User

Retrieve the user created in the previous cell using **two independent paths**:

1. **Via HTTP** — `GET /users/{user_id}` travels through `user_controller.py` → `MCPDBClient.get_user()` → MCP resource `db://users/{user_id}` → DB
2. **Directly via MCP resource** — bypass HTTP entirely and call `db://users/{user_id}` straight from the MCP server

Both paths should return the same user data.

In [13]:
import json
import sys

import requests
from mcp import ClientSession
from mcp.client.stdio import StdioServerParameters, stdio_client
from pydantic import AnyUrl, TypeAdapter


async def fetch_created_user(user_id: int) -> None:
    base = "http://127.0.0.1:8000"

    # --- Path 1: via HTTP → user_controller → MCPDBClient → MCP resource ---
    http_resp = requests.get(f"{base}/users/{user_id}", timeout=10)
    print("=== Path 1: Via HTTP (user_controller → MCPDBClient → db_server) ===")
    print("Status:", http_resp.status_code)
    print("Data:  ", http_resp.json())

    # --- Path 2: directly via MCP resource (bypasses HTTP layer entirely) ---
    params = StdioServerParameters(command=sys.executable, args=["src/db_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.read_resource(
                TypeAdapter(AnyUrl).validate_python(f"db://users/{user_id}")
            )

    print(f"\n=== Path 2: Directly via MCP Resource (db://users/{user_id}) ===")
    mcp_data = None
    if result.contents and hasattr(result.contents[0], "text"):
        mcp_data = json.loads(result.contents[0].text)
        print("Data:", mcp_data)
    else:
        print("No content returned from MCP resource")

    # Verify both paths agree
    http_email = http_resp.json().get("data", {}).get("user", {}).get("email")
    mcp_email = (mcp_data or {}).get("data", {}).get("user", {}).get("email")
    print(f"\nBoth paths return same email: {http_email == mcp_email}  ({http_email})")


await fetch_created_user(created_user_id)

=== Path 1: Via HTTP (user_controller → MCPDBClient → db_server) ===
Status: 200
Data:   {'success': True, 'message': 'Fetched user', 'data': {'user': {'id': 7, 'name': 'Workshop Demo User', 'email': 'workshop.20260315180353@example.com', 'created_at': '2026-03-15T18:03:54.049474Z'}}}

=== Path 2: Directly via MCP Resource (db://users/7) ===
Data: {'success': True, 'message': 'Fetched user', 'data': {'user': {'id': 7, 'name': 'Workshop Demo User', 'email': 'workshop.20260315180353@example.com', 'created_at': '2026-03-15T18:03:54.049474Z'}}}

Both paths return same email: True  (workshop.20260315180353@example.com)


## Part 9) End-to-End Request Flow Diagram

```mermaid
sequenceDiagram
    participant Client as HTTP Client<br/>(notebook / curl)
    participant FastAPI as api_server.py<br/>(FastAPI :8000)
    participant Controller as user_controller.py<br/>(APIRouter /users)
    participant MCPClient as mcp_client.py<br/>(MCPDBClient)
    participant MCPServer as db_server.py<br/>(FastMCP stdio)
    participant DB as PostgreSQL<br/>(:55432)

    Note over Client,DB: ── Create User Flow ──

    Client->>FastAPI: POST /users/create_user {name, email}
    FastAPI->>Controller: create_user(request: CreateUserRequest)
    Controller->>MCPClient: mcp_db_client.create_user(name, email)
    MCPClient->>MCPServer: stdio → call_tool("create_user", {name, email})
    MCPServer->>DB: INSERT INTO users VALUES(...)
    DB-->>MCPServer: RETURNING id, name, email, created_at
    MCPServer-->>MCPClient: {success: true, data: {user: {...}}}
    MCPClient-->>Controller: dict result
    Controller-->>FastAPI: HTTP 200
    FastAPI-->>Client: JSON {success: true, data: {user: {...}}}

    Note over Client,DB: ── Fetch User Flow ──

    Client->>FastAPI: GET /users/{user_id}
    FastAPI->>Controller: get_user(user_id)
    Controller->>MCPClient: mcp_db_client.get_user(user_id)
    MCPClient->>MCPServer: stdio → read_resource("db://users/{user_id}")
    MCPServer->>DB: SELECT * FROM users WHERE id = $1
    DB-->>MCPServer: row
    MCPServer-->>MCPClient: {success: true, data: {user: {...}}}
    MCPClient-->>Controller: dict result
    Controller-->>FastAPI: HTTP 200
    FastAPI-->>Client: JSON {success: true, data: {user: {...}}}

    Note over Client,DB: ── AI Agent Path (db_client.py) ──

    Client->>MCPServer: MCPServerStdio → LLM selects tool/resource
    MCPServer->>DB: Query / mutation
    DB-->>MCPServer: result
    MCPServer-->>Client: JSON output
```

## Conclusion

This session demonstrated the complete MCP request path end-to-end:

| Layer | File | Role |
|-------|------|------|
| MCP Server | `src/db_server.py` | Exposes `@mcp.resource` (read) and `@mcp.tool` (write) over stdio |
| MCP Client | `src/services/mcp_client.py` | Wraps the MCP protocol; spawns `db_server.py` per request |
| HTTP Router | `src/controllers/user_controller.py` | Maps HTTP routes → `MCPDBClient` calls; translates failures to HTTP errors |
| FastAPI App | `src/api_server.py` | Mounts the router; serves on port 8000 |
| Agent Client | `src/db_client.py` | Alternative LLM-powered consumer of the same MCP server |

**Key takeaways:**

- MCP **resources** are read-only and map to `SELECT` queries — identified by URI (`db://users`, `db://users/{id}`)
- MCP **tools** are mutations and map to `INSERT`/`UPDATE`/`DELETE` — called by name with typed arguments
- The HTTP API and the AI agent client are both consumers of the **same** MCP server
- `MCPDBClient` isolates the MCP transport detail so the controller never needs to know about stdio

**Next steps:**

- Add integration tests that assert the same end-to-end path automatically
- Explore MCP prompts as a third surface type alongside resources and tools